# Import

In [1]:
import pandas as pd
import json

from datasets import load_dataset
from datasets import Dataset
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_analyzer.predefined_recognizers import SpacyRecognizer

from projet_14.rgpd import run

c:\Formation\Projet_14\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get hugging face dataset

In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ANR-MALADES/MediQAl", "mcqu")
df = ds["train"].to_pandas()

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


# Analyse dataset

In [3]:
df.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,7543,"Monsieur R. âgé de 37 ans, forestier de son mé...",Au sujet des vaccinations :,Le vaccin contre la fièvre jaune est obligatoire,Le vaccin contre la fièvre jaune peut se faire...,Le vaccin contre le choléra est un vaccin qui ...,Le vaccin contre l'hépatite B est conseillé ma...,Les vaccinations contre le tétanos et la polio...,A,QCU,Infectious Diseases,Understanding
1,10495,NaN,"Devant une tuméfaction cervicale médiane, juxt...",Kyste dermoïde,Laryngocèle,Kyste du tractus thyréoglosse,Grenouillette sublinguale,Lithiase salivaire,C,QCU,Otorhinolaryngology (ENT),Understanding
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding
4,17514,Un homme de 63 ans est hospitalisé pour des do...,Pour confirmer le diagnostic de maladie d'Addi...,Test de stimulation par la 1.24 corticotrophin...,Test de freinage par la dexaméthasone,Test de stimulation de l'activité rénine plasm...,Test de restriction hydrique,Scintigraphie surrénale à l'iodocholestérol,A,QCU,Endocrinology and Metabolism,Reasoning


In [4]:
# Print distinct values of "medical_subject" column
distinct_medical_subjects = df["medical_subject"].unique()
print(f"Distinct values of 'medical_subject' column: {distinct_medical_subjects}")

Distinct values of 'medical_subject' column: <ArrowStringArray>
[             'Infectious Diseases',        'Otorhinolaryngology (ENT)',
     'Endocrinology and Metabolism',                       'Pediatrics',
                     'Microbiology',                      'Orthopedics',
                     'Biochemistry',                       'Cardiology',
        'Gynecology and Obstetrics',                        'Radiology',
                      'Dermatology',                      'Pulmonology',
                       'Psychiatry',                     'Rheumatology',
          'Hepato-Gastroenterology',                       'Hematology',
                        'Neurology',                       'Immunology',
                    'Ophthalmology',                     'Parasitology',
                   'Nephro-Urology',             'Pathological Anatomy',
            'Occupational Medicine', 'Forensic Medicine and Toxicology',
                   'Rehabilitation',                       '

In [5]:
# Keep only the rows where "question_type" is "Reasoning"
# reasoning_df = df[df["question_type"] == "Reasoning"]
# print(f"Number of rows in reasoning_df: {reasoning_df.shape[0]}")

# Keep only the rows where "clinical_case" and "question" are not null in df
df = df.dropna(subset=["clinical_case", "question"])
print(f"Number of rows in reasoning_df after dropping null values: {df.shape[0]}")

Number of rows in reasoning_df after dropping null values: 3935


In [6]:
df.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,7543,"Monsieur R. âgé de 37 ans, forestier de son mé...",Au sujet des vaccinations :,Le vaccin contre la fièvre jaune est obligatoire,Le vaccin contre la fièvre jaune peut se faire...,Le vaccin contre le choléra est un vaccin qui ...,Le vaccin contre l'hépatite B est conseillé ma...,Les vaccinations contre le tétanos et la polio...,A,QCU,Infectious Diseases,Understanding
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding
4,17514,Un homme de 63 ans est hospitalisé pour des do...,Pour confirmer le diagnostic de maladie d'Addi...,Test de stimulation par la 1.24 corticotrophin...,Test de freinage par la dexaméthasone,Test de stimulation de l'activité rénine plasm...,Test de restriction hydrique,Scintigraphie surrénale à l'iodocholestérol,A,QCU,Endocrinology and Metabolism,Reasoning
5,17623,Une femme de 61 ans vient consulter pour asthé...,Parmi les propositions suivantes concernant le...,L'effondrement de la fixation thyroïdienne suf...,Un taux plasmatique de T3 abaissé suffit à aff...,Il n'y a pas d'hypothyroïdie primitive sans él...,La prolactine peut être élevée,La FSH est basse et s'élève sous métopirone,D,QCU,Endocrinology and Metabolism,Understanding


# Build SFT JSONL file from dataset

In [7]:

SYSTEM_PROMPT = "Tu es un assistant médical expert. Réponds de façon précise et concise."


def to_messages(row) -> list[dict]:
    answers = [row[f"answer_{l.strip().lower()}"] for l in row["correct_answers"].split(",")]
    answer = answers[0] if len(answers) == 1 else "\n".join(f"- {a}" for a in answers)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{row['clinical_case']}\n\n{row['question']}"},
        {"role": "assistant", "content": answer},
    ]


df["messages"] = df.apply(to_messages, axis=1)

sft_ds = Dataset.from_pandas(
    df[["id", "medical_subject", "question_type", "messages"]], preserve_index=False
)
sft_ds.to_json("../data/sft_mediqal_mcqu.jsonl", force_ascii=False)
sft_ds[0]["messages"]


Creating json from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 55.90ba/s]


[{'role': 'system',
  'content': 'Tu es un assistant médical expert. Réponds de façon précise et concise.'},
 {'role': 'user',
  'content': "Monsieur R. âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur R. a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr R., de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladies présentes dans la région\n\nAu sujet des vaccinations :"},
 {'role': 'assistant',
  'content': 'Le vaccin contre la fièvre jaune est obligatoire'}]

In [8]:
print(sft_ds)

Dataset({
    features: ['id', 'medical_subject', 'question_type', 'messages'],
    num_rows: 3935
})


# Anonymisation des données

In [9]:
report = run("../data/sft_mediqal_mcqu.jsonl", lang="fr", output_dir="../data/anonymized")

In [10]:
# DataFrame des champs "content" du jsonl anonymisé : une ligne par message (hors system)
rows = []
with open("../data/anonymized/sft_mediqal_mcqu.anonymized.jsonl", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        for message in record["messages"]:
            if message["role"] != "system":
                rows.append({"id": record["id"], "role": message["role"], "content": message["content"]})

contents_df = pd.DataFrame(rows)
contents_df["n_person"] = contents_df["content"].str.count("<PERSON>")
print(f"{len(contents_df)} messages, dont {(contents_df['n_person'] > 0).sum()} avec au moins une balise <PERSON>")
contents_df.head()

7870 messages, dont 1598 avec au moins une balise <PERSON>


,id,role,content,n_person
0,7543,user,"Monsieur <PERSON> âgé de 37 ans, forestier de ...",3
1,7543,assistant,Le vaccin contre la fièvre jaune est obligatoire,0
2,17342,user,Une jeune femme de 24 ans vous consulte pour a...,0
3,17342,assistant,Administration d'antithyroïdiens,0
4,22883,user,"Un nouveau-né de sexe masculin, âgé de 3 jours...",0


In [11]:
import re


def person_contexts(text: str, width: int = 60) -> list[str]:
    """Extrait autour de chaque balise <PERSON>, avec `width` caractères de chaque côté."""
    return [text[max(0, m.start() - width) : m.end() + width].replace("\n", " ") for m in re.finditer("<PERSON>", text)]


# Une ligne par balise <PERSON>, pour vérifier les remplacements
occurrences_df = contents_df[["id", "role"]].copy()
occurrences_df["context"] = contents_df["content"].apply(person_contexts)
occurrences_df = occurrences_df.explode("context").dropna(subset=["context"])

pd.set_option("display.max_colwidth", None)
print(f"{len(occurrences_df)} occurrences de <PERSON>")
occurrences_df.sample(20, random_state=42)

2210 occurrences de <PERSON>


,id,role,context
2898,26859,user,"ique 2/6 associé à un roulement diastolique 2/6, au foyer d’<PERSON>, un souffle diastolique de 1/6.\r TA 100/70 mm Hg.\r RX du"
5412,21257,user,"droite. Avant l'anurie les urines étaient claires. Monsieur <PERSON>.. est en bon état général, apyrétique, avec une pression ar"
5160,16574,user,"Agé de 17 ans, P... <PERSON>, étudiant en terminale D, accuse depuis quelques semaines d"
1468,13683,user,"Mademoiselle <PERSON>.. <PERSON>, âgée de 28 ans. vient de faire une tentative de suicide pa"
5451,20659,assistant,<PERSON>
4942,21676,user,"Monsieur <PERSON> âgé de 78 ans, hypertendu et diabétique est référé par les"
2780,17603,user,"és dont les résultats sont les suivants : - Taux de base en <PERSON> et <PERSON> faibles, avec réponse en <PERSON> supérieure à"
1396,11476,user,"protéinurie par glomérulonéphrite chronique. Cliniquement, <PERSON> a de discrets oedèmes des membres inférieurs et un hydrocèl"
3824,18270,user,"Madame <PERSON>.., âgée de 30 ans, consulte pour des leucorrhées à répétiti"
7240,21782,user,"Monsieur <PERSON> âgé de 62 ans, diabétique, se présente à 13h aux urgences d"
